In [1]:
import numpy as np
import random
from collections import Counter, defaultdict
import os
import time

# =========================
# 1. 参数设置
# =========================
gamma = 0.9
alpha_A = 0.1
alpha_B = 0.1
epsilon = 0.07

num_episodes = 1000
max_steps_per_episode = 100
num_runs = 100

threshold = 30

# =========================
# 自动生成基础随机种子
# =========================
base_seed = int(time.time() * 1000000) % (2**32)
print(f"Base seed for this experiment: {base_seed}")

# =========================
# 保存目录
# =========================
save_dir = "stage1_Q_records"
os.makedirs(save_dir, exist_ok=True)

# =========================
# 2. 状态与行动空间
# =========================
states = ["CC", "CD", "DC", "DD"]
num_states = len(states)

actions = [0, 1]

def action_to_char(a):
    return "C" if a == 1 else "D"

# =========================
# 3. 即时收益矩阵
# =========================
payoff_matrix = {
    ("CC", "CC"): (4, 4), ("CC", "CD"): (1, 5), ("CC", "DC"): (5, 1), ("CC", "DD"): (2, 2),
    ("CD", "CC"): (4, 4), ("CD", "CD"): (1, 5), ("CD", "DC"): (5, 1), ("CD", "DD"): (2, 2),
    ("DC", "CC"): (4, 4), ("DC", "CD"): (1, 5), ("DC", "DC"): (5, 1), ("DC", "DD"): (2, 2),
    ("DD", "CC"): (4, 4), ("DD", "CD"): (1, 5), ("DD", "DC"): (5, 1), ("DD", "DD"): (2, 2)
}

# =========================
# 4. 统计容器
# =========================
strategy_counter = Counter()
manifold_stats = defaultdict(lambda: {
    "count": 0,
    "optimistic": 0,
    "pessimistic": 0,
    "invariant": 0,
    "non_invariant": 0
})

# =========================
# 5. 工具函数
# =========================
def compute_long_term_value(strategy_A, strategy_B, gamma=0.9):
    """
    strategy_A, strategy_B: 例如 [1,0,0,1]，其中 1=C, 0=D
    返回:
        V_A, V_B: 长期价值向量（四个状态各一个）
    """
    P = np.zeros((4, 4))
    R_A = np.zeros(4)
    R_B = np.zeros(4)

    for i, s in enumerate(states):
        aA = strategy_A[i]
        aB = strategy_B[i]
        next_state = ("C" if aA == 1 else "D") + ("C" if aB == 1 else "D")
        j = states.index(next_state)

        P[i, j] = 1
        R_A[i], R_B[i] = payoff_matrix[(s, next_state)]

    I = np.eye(4)
    V_A = np.linalg.inv(I - gamma * P).dot(R_A)
    V_B = np.linalg.inv(I - gamma * P).dot(R_B)
    return V_A, V_B


def compute_non_strategy_avg(Q):
    """
    对一个 Q 表（4x2），每个状态取“非最优动作”的Q值，
    再对4个状态求平均。
    """
    non_strategy_q = []
    for row in Q:
        max_idx = np.argmax(row)
        non_max = np.delete(row, max_idx)
        non_strategy_q.append(np.mean(non_max))
    return np.mean(non_strategy_q)

# =========================
# 6. 重复训练 + 直接统计
# =========================
for run_id in range(num_runs):

    seed = base_seed + run_id
    random.seed(seed)
    np.random.seed(seed)

    Q_A = np.zeros((num_states, 2))
    Q_B = np.zeros((num_states, 2))

    # ===== Q-learning =====
    for episode in range(num_episodes):
        for initial_state in range(num_states):
            state = initial_state

            for step in range(max_steps_per_episode):

                # A 选动作
                if random.random() < epsilon:
                    action_A = random.choice(actions)
                else:
                    candidates = np.where(Q_A[state] == np.max(Q_A[state]))[0]
                    action_A = np.random.choice(candidates)

                # B 选动作
                if random.random() < epsilon:
                    action_B = random.choice(actions)
                else:
                    candidates = np.where(Q_B[state] == np.max(Q_B[state]))[0]
                    action_B = np.random.choice(candidates)

                joint = action_to_char(action_A) + action_to_char(action_B)
                reward_A, reward_B = payoff_matrix[(states[state], joint)]

                next_state = states.index(joint)

                # Q 更新
                Q_A[state, action_A] += alpha_A * (
                    reward_A + gamma * np.max(Q_A[next_state]) - Q_A[state, action_A]
                )
                Q_B[state, action_B] += alpha_B * (
                    reward_B + gamma * np.max(Q_B[next_state]) - Q_B[state, action_B]
                )

                state = next_state

    # =========================
    # 提取策略
    # =========================
    strategy_A = ""
    strategy_B = ""

    for s in range(num_states):
        best_A = np.random.choice(np.where(Q_A[s] == np.max(Q_A[s]))[0])
        best_B = np.random.choice(np.where(Q_B[s] == np.max(Q_B[s]))[0])

        strategy_A += action_to_char(best_A)
        strategy_B += action_to_char(best_B)

    strategy_counter[(strategy_A, strategy_B)] += 1

    # =========================
    # 直接做“第二个程序”的统计
    # =========================
    key = (strategy_A, strategy_B)

    avg_non_strategy_A = compute_non_strategy_avg(Q_A)
    avg_non_strategy_B = compute_non_strategy_avg(Q_B)
    non_strategy_avgs = np.array([avg_non_strategy_A, avg_non_strategy_B])

    # 乐观 / 悲观判定
    pessimistic = np.any(non_strategy_avgs < threshold)   # 只要有一个小于阈值就悲观
    optimistic = np.all(non_strategy_avgs >= threshold)  # 两人都大于等于阈值才乐观

    # 不变流形判定
    strategy_A_binary = np.array([1 if c == "C" else 0 for c in strategy_A])
    strategy_B_binary = np.array([1 if c == "C" else 0 for c in strategy_B])

    V_A, V_B = compute_long_term_value(strategy_A_binary, strategy_B_binary, gamma)

    mean_non_strategy_Q = np.mean(non_strategy_avgs)
    mean_strategy_value = (np.mean(V_A) + np.mean(V_B)) / 2
    invariant = mean_non_strategy_Q < mean_strategy_value
    non_invariant = not invariant

    manifold_stats[key]["count"] += 1
    if optimistic:
        manifold_stats[key]["optimistic"] += 1
    if pessimistic:
        manifold_stats[key]["pessimistic"] += 1
    if invariant:
        manifold_stats[key]["invariant"] += 1
    else:
        manifold_stats[key]["non_invariant"] += 1

    # =========================
    # 输出 run 结果
    # =========================
    print(f"\nRun {run_id + 1:03d} | seed = {seed}")
    print(f"Player A strategy: {strategy_A}")
    print(Q_A)
    print(f"Player B strategy: {strategy_B}")
    print(Q_B)

    print(f"Avg non-strategy Q (A): {avg_non_strategy_A:.4f}")
    print(f"Avg non-strategy Q (B): {avg_non_strategy_B:.4f}")
    print(f"Optimistic: {optimistic}")
    print(f"Pessimistic: {pessimistic}")
    print(f"Invariant: {invariant}")

    # =========================
    # 保存 Q 表
    # =========================
    filename = f"{save_dir}/run_{run_id+1:03d}.npz"
    np.savez(
        filename,
        Q_A=Q_A,
        Q_B=Q_B,
        strategy_A=strategy_A,
        strategy_B=strategy_B,
        seed=seed,
        base_seed=base_seed,
        avg_non_strategy_A=avg_non_strategy_A,
        avg_non_strategy_B=avg_non_strategy_B,
        optimistic=optimistic,
        pessimistic=pessimistic,
        invariant=invariant,
        non_invariant=non_invariant
    )

# =========================
# 7. 输出第一部分统计：策略频次
# =========================
print("\n=== Strategy Statistics ===")
print(f"Base seed = {base_seed}")
for (sA, sB), count in strategy_counter.most_common():
    print(f"Player A: {sA}, Player B: {sB} → count: {count}")

# =========================
# 8. 输出第二部分统计：策略流形统计（按总数从高到低）
# =========================
print("\n=== Manifold Statistics (sorted by count) ===")
for key, stats in sorted(manifold_stats.items(), key=lambda x: x[1]["count"], reverse=True):
    print(f"策略流形 {key}:")
    print(f"  总数: {stats['count']}")
    print(f"  乐观: {stats['optimistic']}")
    print(f"  悲观: {stats['pessimistic']}")
    print(f"  不变流形: {stats['invariant']}")
    print(f"  非不变流形: {stats['non_invariant']}")
    print()

Base seed for this experiment: 1568985886

Run 001 | seed = 1568985886
Player A strategy: CDDC
[[37.4241122  38.80075523]
 [36.47901578 32.9817902 ]
 [36.4999654  33.26259659]
 [35.89564268 38.32423064]]
Player B strategy: CDDC
[[37.34619675 38.57723044]
 [36.22772005 33.34816406]
 [36.26770527 33.09556638]
 [35.93585039 38.27472092]]
Avg non-strategy Q (A): 34.8910
Avg non-strategy Q (B): 34.9314
Optimistic: True
Pessimistic: False
Invariant: True

Run 002 | seed = 1568985887
Player A strategy: CDDC
[[37.33422311 38.34056666]
 [36.46487287 33.14442553]
 [36.49159449 33.30607262]
 [35.73476863 38.60402183]]
Player B strategy: CDDC
[[37.36858502 38.67076523]
 [36.48249158 33.34511655]
 [36.55141074 33.57614707]
 [36.17891466 38.52033793]]
Avg non-strategy Q (A): 34.8799
Avg non-strategy Q (B): 35.1172
Optimistic: True
Pessimistic: False
Invariant: True

Run 003 | seed = 1568985888
Player A strategy: CDDC
[[37.02562395 38.12967555]
 [36.26946437 33.1308702 ]
 [36.15649916 33.03415229]
 [

KeyboardInterrupt: 